# M5L2 - Data Fetching and Organization

In this Notebook, we will create a local environment for the use of `STAR`, including other tools needed for RNA-seq genome alignment, and prepare our project folder. This environment can be used for the final project if needed. 

Objectives: 
- Create the module_5_STAR environment locally
- Create the `m5_project_STAR` folder for use this week
- retrieve samples from [GSE128157](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE128157)
- Run Pre-Processing (QC checks + trimming with `fastp`) on these samples

#### [Lecture M5L2V2](https://youtu.be/lVIKdR88nL0)

```{note}
**Runs on an HPC.** This notebook was written for the Hellbender HPC cluster (University of Missouri) and uses SLURM and/or large sequencing files. It is shown here for reading; saved outputs were removed for privacy. Set `REPO_DIR`, `DATA_DIR` and `WORK_DIR` (see the repository README) and download inputs with `data/fetch_data.sh` before running it on your own cluster.
```

## Sample data 

Once again, we are investigating the data set we have used in previous weeks. This time, though, we will retrieve 12 samples from both sets of experiments within the SuperSet [GSE128157](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE128157). These 12 samples are the first technical replicates of the *A. thaliana* roots for both Col-0 and OPT3 mutants in replete and 72hr Cd treatment. We are doing this so we can run `STAR` on these samples and align to the genome using a traditional alignment. This will produce `bam` files we can use in IGV, and next week we will compare this traditional alignment to the Tuxedo protocol and our pseudo-alignment using salmon from Module 3 and Module 4. The samples are listed in a table below: 

| Sample Name | SRX Accession | Genotype | Tissue | Treatment | Bio Rep | Tech Rep |
|-------------|---------------|----------|--------|-----------|---------|----------|
| ORT3-opt3-2_root_treated_3-1 | SRX5507654 | OPT3-2 | Root | 72h Cd (T) | 3 | 1 |
| ORT2-opt3-2_root_treated_2-1 | SRX5507651 | OPT3-2 | Root | 72h Cd (T) | 2 | 1 |
| ORT1-opt3-2_root_treated_1-1 | SRX5507648 | OPT3-2 | Root | 72h Cd (T) | 1 | 1 |
| CRT3-Col_root_treated_3-1 | SRX5507636 | Col-0 | Root | 72h Cd (T) | 3 | 1 |
| CRT2-Col_root_treated_2-1 | SRX5507633 | Col-0 | Root | 72h Cd (T) | 2 | 1 |
| CRT1-Col_root_treated_1-1 | SRX5507630 | Col-0 | Root | 72h Cd (T) | 1 | 1 |
| ORN3-1 | SRX1634999 | OPT3-2 | Root | Replete (N) | 3 | 1 |
| ORN2-1 | SRX1634996 | OPT3-2 | Root | Replete (N) | 2 | 1 |
| ORN1-1 | SRX1634993 | OPT3-2 | Root | Replete (N) | 1 | 1 |
| CRN3-1 | SRX1634981 | Col-0 | Root | Replete (N) | 3 | 1 |
| CRN2-1 | SRX1634978 | Col-0 | Root | Replete (N) | 2 | 1 |
| CRN1-1 | SRX1634975 | Col-0 | Root | Replete (N) | 1 | 1 |


# Creation of the environment

We will create a local environment called `module_5_STAR` to use within `mamba` for some of the labs this week. The enviornments `.yml` file is located in `/Module_5/Resources/module_5_STAR.yml`. This environment contains the following packages:

| Tool | Purpose / Description |
|------|---------------------|
| **fastp** | Quality control and trimming of raw FASTQ reads; removes adapters, filters low-quality reads, and generates QC reports. |
| **multiqc** | Aggregates QC reports from multiple samples/tools (like fastp, Qualimap) into a single summary report. |
| **STAR** | Splice-aware aligner for mapping RNA-Seq reads to a reference genome; supports 1-pass and 2-pass mapping. |
| **samtools** | Utilities for working with SAM/BAM files: sorting, indexing, filtering, and basic statistics. |
| **deeptools** | Tools for processing and visualizing genome-wide sequencing data; includes coverage plots, heatmaps, and normalization. |
| **salmon** | Pseudo-aligner for fast transcript-level quantification from RNA-Seq reads without full genome alignment. |
| **sra-tools** | Utilities for downloading and converting sequencing data from the NCBI SRA database (e.g., `fastq-dump`). |
| **gffread** | Parses and converts GFF/GTF annotation files; can extract transcript sequences from a reference genome. |
| **bedtools** | Suite of tools for genome arithmetic: intersecting, merging, and manipulating BED/GFF files. |
| **subread** | Includes `featureCounts` for counting reads mapped to genes or exons; used for generating count matrices. |
| **entrez-direct** | Command-line tools for querying NCBI databases (e.g., for metadata retrieval or batch downloads). |
| **ucsc-wigtobigwig** | Converts UCSC WIG files to BigWig format for efficient visualization in genome browsers. |
| **pigz** | Parallelized gzip for fast compression/decompression of large sequencing files. |
| **qualimap** | Evaluates alignment quality; produces metrics like coverage, GC content, and mapping bias reports. |

In [ ]:
# RUN EVERY TIME YOU USE BASH IN JUPYTER
export TERM=xterm

In [ ]:
ml mamba

mamba env create -f ../../../environment/module_5_star.yml


# If you have already created the environment `module_5_STAR` run this to run the rest of the notebook. Each time the kernel restarts run the cell below: 

In [ ]:
export TERM=xterm
ml mamba

source activate module_5_STAR

Check our current working directory, should be `XXX/Module_5/Lab`

In [ ]:
pwd

We will now create our project files to look like this: 

    m5_project_STAR/
    ├── data
    │   ├── raw
    │   └── trimmed
    ├── references
    │   ├── annotation
    │   ├── genome
    │   ├── sra_info
    │   └── star_index
    ├── results
    │   ├── alignments
    │   ├── counts
    │   └── qc
    └── scripts

Remember, proper folder organization is important for use in RNA-Seq processing. Learn more about it [here](https://hbctraining.github.io/Training-modules/planning_successful_rnaseq/slides/data_management_2019.pdf)

In [ ]:
#create project files: 

cd ..

pwd

mkdir -p m5_project_STAR/{data/{raw,trimmed},references/{genome,annotation,star_index,sra_info},results/{alignments,qc,counts},scripts}

tree m5_project_STAR/

cd m5_project_STAR

pwd

We will now retreive the SRA run info for our SuperSet GSE128157

In [ ]:
esearch -db gds -query "GSE128157" | efetch -format runinfo > references/sra_info/GSE128157_runinfo.txt

We now want to pull the info from our 12 samples. 

The file `GSE128157_runinfo.txt` looks like this: 

![image.png](images/M5L2_Data_Fetching_and_Organization_604a6b44.png)

So we parse it using `awk` to generate a csv called `GSE128157_selected_samples.csv` file containing this information:

![image.png](images/M5L2_Data_Fetching_and_Organization_8d0eaae8.png)


In [ ]:
cd Module_5
cd m5_project_STAR
pwd

In [ ]:
# Set the 12 samples we want (full names as in the run info file)
SAMPLES="CRT1-Col_root_treated_1-1 CRT2-Col_root_treated_2-1 CRT3-Col_root_treated_3-1 ORT1-opt3-2_root_treated_1-1 ORT2-opt3-2_root_treated_2-1 ORT3-opt3-2_root_treated_3-1 CRN1-1 CRN2-1 CRN3-1 ORN1-1 ORN2-1 ORN3-1"

# Parse the GSE128157_runinfo.txt file for Sample name, GSM accession, and SRX
awk -v samples="$SAMPLES" '
BEGIN {
    split(samples, a, " ")
    print "sample,GSM,SRX"
}
/Sample[[:space:]]+Accession:/ { gsm=$3 }
/SRA Run Selector:/ {
    match($0,/SRX[0-9]+/,srx)
    if(name_in_list) { print name_in_list "," gsm "," srx[0] }
    name_in_list=""
}
/^[0-9]+\. / {
    name_in_list=""
    line=$0
    for(i in a) if(index(line, a[i])) { name_in_list=a[i]; break }
}
' references/sra_info/GSE128157_runinfo.txt > references/sra_info/GSE128157_selected_samples.csv


We now need to find the SRR IDs associated with our sample's SRX ID. This is needed as some SRX's would have multiple runs, but because our data is SE reads, we will only have 1 SRR per an SRX ID accession. We use the `esearch` command for this. 

In [ ]:
mkdir -p data/raw
CSV="references/sra_info/GSE128157_selected_samples.csv"
TMP="${CSV}.tmp"

# Open the CSV on fd 3
exec 3< "$CSV"

# Read and skip the original header
read -r header <&3

# Write new header to TMP
echo "${header},SRR" > "$TMP"

while IFS=',' read -r sample gsm srx <&3; do
    echo "Processing $sample ($srx)..."
    
    # Fetch SRR IDs
    mapfile -t srrs < <(esearch -db sra -query "$srx" | efetch -format runinfo | tail -n +2 | cut -d, -f1)
    srr=$(printf "%s;" "${srrs[@]}")
    srr=${srr%;} 

    # Append row with SRR
    echo "${sample},${gsm},${srx},${srr}" >> "$TMP"

done

# Close fd 3
exec 3<&-

mv "$TMP" "$CSV"

echo "Updated $CSV with SRR column."

Now we will use `parallel` to download our `fastq` files simultaniously. This should be quite a bit faster than last time, but we are still downloading ~48GB of files (which will compress down to ~12GB with `pigz` parralell `gzip` compression) so it will probably take about 30 minutes to run. **LEAVE JUPYTER OPEN WHILE THIS RUNS!**

In [ ]:
set -euo pipefail

CSV="references/sra_info/GSE128157_selected_samples.csv"
RAW_DIR="data/raw"
mkdir -p "$RAW_DIR"
N_JOBS=8  # number of parallel downloads

download_srr() {
    local sample="$1"
    local srr="$2"

    if [[ -f "$RAW_DIR/${srr}.fastq.gz" ]]; then
        echo "Skipping $sample ($srr), already downloaded."
        return
    fi

    echo "Downloading $srr for $sample..."

    fasterq-dump "$srr" --outdir "$RAW_DIR" --split-files --threads 2

    for fq in "$RAW_DIR/${srr}"*.fastq; do
        [[ -f "$fq" ]] || continue
        pigz -p 4 -f "$fq"      
        gzfile="${fq}.gz"        
        if [[ "$gzfile" == *"_1.fastq.gz" ]]; then
            mv "$gzfile" "$RAW_DIR/${sample}_R1.fastq.gz"
        elif [[ "$gzfile" == *"_2.fastq.gz" ]]; then
            mv "$gzfile" "$RAW_DIR/${sample}_R2.fastq.gz"
        else
            mv "$gzfile" "$RAW_DIR/${sample}_R1.fastq.gz"   # single-end fallback
        fi
    done
}

export -f download_srr
export RAW_DIR

tail -n +2 "$CSV" | while IFS=',' read -r sample gsm srx srr; do
    echo "$sample,$srr"
done | parallel -j "$N_JOBS" --colsep ',' download_srr {1} {2}

wait
echo "All downloads finished."


We now run `fastp` on all of our downloaded samples within the `m5_project_star/data/raw` folder. This will output the qc results to the folder `m5_project_STAR/"results/qc/trimmed_fastp/${base}_fastp_out.json"`. 

Take a look at the results, and add your comments in the text box below:

In [ ]:
mkdir -p results/qc/trimmed_fastp/

for file in data/raw/*.fastq.gz
do
  base=$(basename "$file" .fastq.gz)
  fastp \
    -i "$file" \
    -o "data/trimmed/${base}_trimmed.fastq.gz" \
    --html "results/qc/trimmed_fastp/${base}_fastp_out.html" \
    --json "results/qc/trimmed_fastp/${base}_fastp_out.json" \
    --thread 8 \
    --detect_adapter_for_pe \
    --trim_front1 5 \
    --cut_tail \
    --length_required 30
done